# Full factorials: replicates, center points, and three factors

**Source worksheets:** [yint.org/w3](https://yint.org/w3) and [yint.org/w4](https://yint.org/w4) - weeks 3 and 4 of the applied DoE short course.

Modules 1 and 2 stopped at the four corners of a 2x2.  In a real
project you do three things on top of that, often before the experiment
even finishes:

1. **Replicate** corners so you can read the noise level, get a real
   confidence interval, and stop kidding yourself with ``R^2 = 1``.
2. Run a **center point** at the middle of the design to check whether
   the response surface is genuinely a plane, or whether it is starting
   to curve.
3. Add a **third factor**.  The cube plot is back; the same toolkit
   scales without any new mathematics.

Module 3 works through three case studies that hit each of these in
turn: a chemical side-product 2x2 with a center point (w3), an online
shop 2x2 with replicates (w4), and a 3-factor stability study where
one factor turns out to barely matter (w4).

## A. Side-product factorial with a center point (w3)

A factorial experiment is run to **minimize** an unwanted side product
in a chemical process.  Two factors are varied:

- **A = additive volume** [20 or 30 mL]
- **B = baffles in the reactor?** [No or Yes]

| Run | A (mL) | B (baffles) | Side product (g) |
|----:|------:|------------:|-----------------:|
|  1  |   20  |     No      |        89        |
|  2  |   30  |     No      |       268        |
|  3  |   20  |    Yes      |       179        |
|  4  |   30  |    Yes      |       448        |

The worksheet asks for the prediction equation, an interpretation of
each coefficient, a comment on the interaction, and where to run the
next experiment to *minimize* the side product.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

from process_improve.experiments import c, gather, lm, main_effects_plot, predict
from process_improve.experiments.visualization import visualize_doe

pio.renderers.default = "notebook_connected"

# Standard order: A, B coded.  Categorical B uses No=-1, Yes=+1.
A = c(-1, +1, -1, +1, lo=20, hi=30, name="A", units="mL")
B = c(-1, -1, +1, +1, lo=0,  hi=1,  name="B", units="0=No, 1=Yes")
y = c(89, 268, 179, 448, name="y")
side = gather(A=A, B=B, y=y)
side

In [ ]:
model = lm("y ~ A + B + A:B", side)
params = model.get_parameters(drop_intercept=False)
print(params.to_string())
print()
print("Prediction equation:")
print(f"  y = {params['Intercept']:.0f} "
      f"+ {params['A']:+.0f} * x_A "
      f"+ {params['B']:+.0f} * x_B "
      f"+ {params['A:B']:+.0f} * x_A * x_B")

In [ ]:
sq = visualize_doe(
    plot_type="square_plot",
    design_data=side.to_dict(orient="records"),
    response_column="y",
    factors_to_plot=["A", "B"],
    factor_labels={"A": "Additive [mL]", "B": "Baffles"},
    backend="plotly",
)
fig = go.Figure(sq["plotly"])
fig.update_layout(width=520, height=440, title="Side product at the four corners")
fig

### Adding two center-point runs

Two extra experiments are run at the *center* of the additive range
(25 mL), once with each level of baffles:

| Run | A (mL) | B (baffles) | Side product (g) |
|----:|------:|------------:|-----------------:|
|  5  |   25  |     No      |       186        |
|  6  |   25  |    Yes      |       300        |

These do not change which terms are in the model, but they let us check
whether the linear-and-interaction model is actually the right model.
If the center-point responses match the model's predictions at
``x_A = 0``, the surface is linear in A.  If they sit well off the
plane, the system has *curvature* and a quadratic term is needed.

In [ ]:
A2 = c(-1, +1, -1, +1,  0,  0, lo=20, hi=30, name="A", units="mL")
B2 = c(-1, -1, +1, +1, -1, +1, lo=0,  hi=1,  name="B", units="0=No, 1=Yes")
y2 = c(89, 268, 179, 448, 186, 300, name="y")
side2 = gather(A=A2, B=B2, y=y2)

model2 = lm("y ~ A + B + A:B", side2)
params2 = model2.get_parameters(drop_intercept=False)
print("Coefficients with the center points included:")
print(params2.to_string())
print()
print("Predicted vs observed at the center points:")
for label, point, observed in [
    ("(x_A=0, x_B=-1)", dict(A=0, B=-1), 186),
    ("(x_A=0, x_B=+1)", dict(A=0, B=+1), 300),
]:
    pred = float(predict(model2, **point).iloc[0])
    print(f"  {label}: predicted {pred:6.1f}, observed {observed} - diff {observed - pred:+.1f}")

## B. Online shop 2x2 with replicates (w4)

A small online business runs an 8-week experiment every Tuesday (same
day of week, to remove the day-of-week effect).  Two factors:

- **S = free-shipping threshold** [€30 or €50]
- **P = profile required before checkout?** [No or Yes]

Each corner is *replicated*: two Tuesdays per combination, eight
Tuesdays total.

| S (euro) | P (profile) | Daily sales (euro) |
|---------:|------------:|-------------------:|
|    30    |     No      |   348   and   356  |
|    50    |     No      |   359   and   363  |
|    30    |    Yes      |   327   and   296  |
|    50    |    Yes      |   243   and   257  |

Now ``R^2`` is no longer exactly 1 - the replicates give the regression
something to disagree with - and you can read the *noise level* off
the residuals.

In [ ]:
import numpy as np

S = c(-1, -1, -1, -1, +1, +1, +1, +1, lo=30, hi=50, name="S", units="euro")
P = c(-1, -1, +1, +1, -1, -1, +1, +1, lo=0,  hi=1,  name="P", units="0=No, 1=Yes")
sales = c(348, 356, 327, 296, 359, 363, 243, 257, name="sales")
shop = gather(S=S, P=P, sales=sales)

m = lm("sales ~ S + P + S:P", shop)
params = m.get_parameters(drop_intercept=False)
print(params.to_string())
print()
# Estimate the noise level from the regression residuals.
fitted = shop.apply(lambda r: float(predict(m, S=r["S"], P=r["P"]).iloc[0]), axis=1)
residuals = shop["sales"] - fitted
print(f"Standard error of the residuals: {np.std(residuals, ddof=4):.1f} euro")
print(f"Range across replicates at each corner: "
      f"S- P- = {356-348} | S+ P- = {363-359} | "
      f"S- P+ = {327-296} | S+ P+ = {257-243}")

In [ ]:
fig = main_effects_plot(m, factor_labels={"S": "Free-ship threshold", "P": "Profile required"})
fig.update_layout(width=620, height=380, title="Online shop main effects")
fig

### Q6: a mistake on the last experiment

The colleague running the last Tuesday set the shipping threshold to
€60 by accident (intended: €50) and recorded €220 (instead of €257).
Two changes: the factor level is now at ``x_S = +2`` rather than
``+1``, and the response is different.

Rule of thumb from the source material: **record what really happened,
not what was supposed to happen.**  The model accommodates the odd
level naturally.

In [ ]:
S2 = c(-1, -1, -1, -1, +1, +1, +1, +2, lo=30, hi=50, name="S", units="euro")
P2 = c(-1, -1, +1, +1, -1, -1, +1, +1, lo=0,  hi=1,  name="P", units="0=No, 1=Yes")
sales2 = c(348, 356, 327, 296, 359, 363, 243, 220, name="sales")
shop2 = gather(S=S2, P=P2, sales=sales2)
m2 = lm("sales ~ S + P + S:P", shop2)
print("Coefficients (with the mistake recorded honestly):")
print(m2.get_parameters(drop_intercept=False).to_string())

## C. A three-factor stability study (w4)

A development team is trying to push product stability above 50 days
with three factors:

- **A = enzyme strength** [20% or 30%]
- **B = feed concentration** [75% or 85%]
- **C = mixer type** [R or W]  (categorical)

| A   | B   | C   | Stability (days) |
|----:|----:|----:|-----------------:|
|  -  |  -  |  -  |        40        |
|  +  |  -  |  -  |        27        |
|  -  |  +  |  -  |        35        |
|  +  |  +  |  -  |        21        |
|  -  |  -  |  +  |        41        |
|  +  |  -  |  +  |        27        |
|  -  |  +  |  +  |        31        |
|  +  |  +  |  +  |        20        |

The worksheet asks which factors matter, what to do with the one that
does not, and where to run the next experiments to reach 50 days.

In [ ]:
A = c(-1, +1, -1, +1, -1, +1, -1, +1, lo=20, hi=30, name="A", units="%")
B = c(-1, -1, +1, +1, -1, -1, +1, +1, lo=75, hi=85, name="B", units="%")
C = c(-1, -1, -1, -1, +1, +1, +1, +1, lo=0,  hi=1,  name="C", units="0=R, 1=W")
stab = c(40, 27, 35, 21, 41, 27, 31, 20, name="stab")
study = gather(A=A, B=B, C=C, stab=stab)

m_full = lm("stab ~ A * B * C", study)
print("Full model (all main effects and interactions):")
print(m_full.get_parameters(drop_intercept=False).to_string())

In [ ]:
# A three-factor cube plot puts the eight observations on the vertices.
cube = visualize_doe(
    plot_type="cube_plot",
    design_data=study.to_dict(orient="records"),
    response_column="stab",
    factors_to_plot=["A", "B", "C"],
    factor_labels={"A": "Enzyme [%]", "B": "Feed [%]", "C": "Mixer"},
    backend="plotly",
)
fig = go.Figure(cube["plotly"])
fig.update_layout(width=560, height=480, title="Three-factor stability cube")
fig

In [ ]:
# Pareto-style bar chart of the effect magnitudes.
pareto = visualize_doe(
    plot_type="pareto",
    analysis_results={
        "effects": {
            term: 2 * coef
            for term, coef in m_full.get_parameters(drop_intercept=True).items()
        },
    },
    backend="plotly",
)
fig = go.Figure(pareto["plotly"])
fig.update_layout(width=640, height=380, title="Pareto plot of effects on stability")
fig

In [ ]:
# Drop the C main effect (smallest in magnitude) and refit.
m_reduced = lm("stab ~ A * B", study)
print("Reduced model (C dropped):")
print(m_reduced.get_parameters(drop_intercept=False).to_string())

In [ ]:
# Three suggested next experiments to reach 50 days.
candidates = [
    ("A=-2 (15%), B=-2 (70%), C=0",  dict(A=-2,   B=-2)),
    ("A=-3 (10%), B=-1 (75%), C=0",  dict(A=-3,   B=-1)),
    ("A=-2.5 (12.5%), B=-1.5 (72.5%), C=0", dict(A=-2.5, B=-1.5)),
]
for label, coded in candidates:
    pred = float(predict(m_reduced, **coded).iloc[0])
    print(f"{label:50s} -> predicted stability {pred:5.1f} days")

## Wrap-up

Three case studies in one module, each adding one trick to the
toolbox:

- **Center points** confirm whether the design plane is straight.
- **Replicates** give a real noise estimate (and a real ``R^2``).
- **Three factors** fit on a cube plot; the prediction equation grows
  by two main-effect terms and three interactions, but the workflow
  is identical to a 2x2.

**Next:** Module 4 scales the same toolkit to **four factors** in a
bioreactor study (16 runs, full factorial) and walks through how to
read a Pareto / half-normal plot and *prune* a model down to the
terms that actually matter.